# Road Following Live (ONNX + ROS Topic + Video Recording)

This notebook subscribes to the **ROS Camera Topic** (`/csi_cam_0/image_raw`), executing ONNX model inference and **Stanley Control** on physical JetRacer.
It records the processed driving video directly to an **MP4 video file** (`output_drive.mp4`) for offline viewing without display overhead.

### 1. Setup Environment & Load ONNX Model

In [ ]:
import os
import sys
from pathlib import Path

# Add parent directory to sys.path to access Controller.py, Runner.py, and utils.py
parent_dir = Path.cwd().parent
if str(parent_dir) not in sys.path:
    sys.path.append(str(parent_dir))

import onnxruntime as ort

# Locate ONNX model file
model_path = os.path.join(Path.cwd(), "road_following_model.onnx")
if not os.path.exists(model_path):
    model_path = os.path.join(parent_dir, "notebooks", "road_following_model.onnx")

if not os.path.exists(model_path):
    print(f"[!] ERROR: ONNX model file '{model_path}' not found!")
else:
    print(f"[*] Loading ONNX model from: {model_path}")

available_providers = ort.get_available_providers()
providers = ['CUDAExecutionProvider'] if 'CUDAExecutionProvider' in available_providers else []
providers.append('CPUExecutionProvider')

try:
    session = ort.InferenceSession(model_path, providers=providers)
except Exception:
    session = ort.InferenceSession(model_path, providers=['CPUExecutionProvider'])

input_name = session.get_inputs()[0].name
output_name = session.get_outputs()[0].name
print(f"[+] Loaded ONNX Session with providers: {session.get_providers()}")


### 2. Initialize ROS Node & JetRacer Hardware (`NvidiaRacecar`)

In [ ]:
import rospy
from sensor_msgs.msg import Image as ROSImage
from jetracer.nvidia_racecar import NvidiaRacecar
try:
    from jetracer.Controller import StanleyController
    from jetracer.Runner import JetRacerROSOnnxRunner
except ImportError:
    from Controller import StanleyController
    from Runner import JetRacerROSOnnxRunner

# 1. Initialize ROS Node
try:
    rospy.init_node('road_following_live_notebook', anonymous=True, disable_signals=True)
    print("[+] ROS Node initialized successfully!")
except Exception as e:
    print(f"[*] ROS Node notice: {e}")

# 2. Hardware & Controller Setup
car = NvidiaRacecar()
stanley = StanleyController()
stanley.reset()
print("[+] JetRacer hardware and Stanley Controller initialized.")


### 3. Setup ROS Subscriber & Video Recording Runner

In [ ]:
video_output_path = os.path.join(Path.cwd(), "output_drive.mp4")

runner = JetRacerROSOnnxRunner(
    session=session,
    input_name=input_name,
    output_name=output_name,
    car=car,
    stanley=stanley,
    k=2.5,
    throttle=0.20,
    brake_gain=0.10,
    bias=0.0,
    alpha=0.4,
    video_path=video_output_path,
    video_fps=20.0
)

runner.running = False  # Start paused

topic_name = "/csi_cam_0/image_raw"
ros_sub = rospy.Subscriber(topic_name, ROSImage, runner.image_callback, queue_size=1, buff_size=2**24)
print(f"[*] Subscribed to ROS Image Topic: {topic_name}")
print(f"[*] Video recording configured -> {video_output_path}")


### 4. Run Autonomous Driving & Record Video File

In [ ]:
# Run this cell to START driving and recording video!
# Press Stop/Interrupt kernel button to stop car and save the video file.

import time

runner.running = True
stanley.reset()
print("=======================================================")
print("   AUTONOMOUS DRIVING & VIDEO RECORDING ACTIVE         ")
print(f"   Saving video to: {video_output_path}               ")
print("   Press Interrupt/Stop kernel button to stop car.     ")
print("=======================================================\n")

try:
    while runner.running:
        time.sleep(0.1)
except KeyboardInterrupt:
    pass
finally:
    runner.stop()


### 5. Emergency Stop Cell

In [ ]:
# Emergency Stop Cell
runner.stop()
